**Live Demo Link:** PASTE_YOUR_STREAMLIT_LINK_HERE

---
# Assignment 2 – Heart Disease Prediction System
**Course:** Machine Learning Fundamentals (AIC354) | **Instructor:** Dr. Jawad Shafi

**Goal:** Build a Heart Disease Prediction system using only **4 input attributes**, following the same steps as the Titanic system.

**Four input attributes chosen:**
| Attribute | Meaning |
|---|---|
| `age` | Age of the patient (years) |
| `sex` | Male / Female |
| `cp` | Chest pain type (1 to 4) |
| `thalach` | Maximum heart rate achieved |

**Target:** `target` -> 0 = No heart disease, 1 = Heart disease

## Phase 1: Data Collection
Dataset: UCI Heart Disease (Cleveland file `processed.cleveland.data`). It has no header row, so we give the column names ourselves.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

columns = ['age','sex','cp','trestbps','chol','fbs','restecg',
           'thalach','exang','oldpeak','slope','ca','thal','num']

df = pd.read_csv('processed.cleveland.data', header=None, names=columns, na_values='?')
df.head()

## Phase 2: Data Understanding

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Original target values (num):")
print(df['num'].value_counts())

**What we see:** In the original data, `num` is 0 (healthy) or 1, 2, 3, 4 (different levels of heart disease). We only want a Yes/No answer, so we will turn 1-4 into **1**.

## Phase 3: Data Preprocessing

**Step 3.1 - Keep only 4 input attributes + the target**

In [ ]:
data = df[['age', 'sex', 'cp', 'thalach', 'num']].copy()
data.head()

**Step 3.2 - Make the target Yes/No (0 or 1)**

In [ ]:
data['target'] = (data['num'] > 0).astype(int)
data = data.drop(columns=['num'])
print(data['target'].value_counts())

**Step 3.3 - Handle missing values**

In [ ]:
print("Missing values before:", data.isnull().sum().sum())
data = data.dropna()
print("Missing values after :", data.isnull().sum().sum())
print("Rows left:", len(data))

**Step 3.4 - Label Encoding**

Models understand numbers, not words. In this dataset `sex` is stored as 1/0, so first we write it as words (Male/Female), then use `LabelEncoder` to turn the words back into numbers (same as the Titanic `Sex` column).

In [ ]:
data['sex'] = data['sex'].map({1: 'Male', 0: 'Female'})

le_sex = LabelEncoder()
data['sex'] = le_sex.fit_transform(data['sex'])

print("Encoding used:", dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_))))
data.head()

## Phase 4: Model Training and Evaluation

**Step 4.1 - Split into Inputs (X) and Output (y), then Train/Test split (80% / 20%)**

In [ ]:
X = data[['age', 'sex', 'cp', 'thalach']]
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Training rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])

**Step 4.2 - Train the model (Random Forest)**

In [ ]:
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
print("Model trained!")

**Step 4.3 - Evaluate the model on unseen test data**

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values()
importance.plot(kind='barh', title='Which input matters most?')
plt.show()

## Save the trained model (needed for deployment)

In [ ]:
joblib.dump(model, 'heart_model.pkl')
joblib.dump(le_sex, 'sex_encoder.pkl')
print("Saved: heart_model.pkl and sex_encoder.pkl")

**Test the saved model like a real user would**

In [ ]:
loaded_model = joblib.load('heart_model.pkl')
loaded_le = joblib.load('sex_encoder.pkl')

# Example patient: 55 years old, Male, chest pain type 4, max heart rate 130
sex_num = loaded_le.transform(['Male'])[0]
sample = pd.DataFrame([[55, sex_num, 4, 130]], columns=['age', 'sex', 'cp', 'thalach'])

pred = loaded_model.predict(sample)[0]
prob = loaded_model.predict_proba(sample)[0][1]
print("Prediction:", "Heart Disease" if pred == 1 else "No Heart Disease")
print("Chance of disease:", round(prob * 100, 1), "%")

## Phase 5: Cloud Deployment
The model is deployed with **Streamlit Community Cloud** using the files `app.py`, `requirements.txt`, `heart_model.pkl` and `sex_encoder.pkl` stored in a GitHub repository.

**Live demo link:** PASTE_YOUR_STREAMLIT_LINK_HERE

## Conclusion
- Followed the same steps as the Titanic system: collect -> understand -> preprocess (with label encoding) -> train -> evaluate -> deploy.
- Used only 4 inputs: age, sex, chest pain type, maximum heart rate.
- The model was saved and deployed on the cloud so anyone can use it from a web link.